# P6-3 · Quantization và inference nhanh

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

## Bài tập 4: Lượng tử hóa int8 một ma trận trọng số {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
W = rng.normal(0, 0.02, (256, 256))
W[3, 10], W[200, 5] = 1.5, -1.2                       # hai giá trị ngoại lai
x = rng.normal(size=(32, 256))

def quant_int8(W, per_channel):
    # TODO: trả về (W giải lượng tử, scale)
    return W, 1.0

In [ ]:
def quant_int8(W, per_channel):
    m = np.abs(W).max(axis=1, keepdims=True) if per_channel else np.abs(W).max()
    scale = m / 127
    q = np.clip(np.round(W / scale), -127, 127).astype(np.int8)
    return q.astype(np.float64) * scale, scale

y = x @ W.T
for pc in (False, True):
    Wq, _ = quant_int8(W, pc)
    print(f"{'từng hàng' if pc else 'cả tensor':9s}: sai số trọng số {np.abs(W - Wq).mean():.2e}, "
          f"sai số tương đối đầu ra {np.linalg.norm(x @ Wq.T - y) / np.linalg.norm(y):.2%}")

cả tensor: sai số trọng số 2.95e-03, sai số tương đối đầu ra 15.95%
từng hàng: sai số trọng số 1.39e-04, sai số tương đối đầu ra 1.41%


## Bài tập 5: KV cache: cùng kết quả, ít tính toán hơn {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch
torch.manual_seed(0)
d = 64
Wq, Wk, Wv = (torch.randn(d, d) / d ** 0.5 for _ in range(3))
prompt = torch.randn(100, d)
moi = torch.randn(50, d)                 # thay cho embedding của token được sinh ra

def attn_cuoi(q, K, V):
    return torch.softmax(q @ K.T / d ** 0.5, dim=-1) @ V
# TODO: cách (a) không cache; cách (b) có cache; dem_a, dem_b; so out_a, out_b

In [ ]:
dem_a = 0
for i in range(50):
    X = torch.cat([prompt, moi[: i + 1]])
    K, V = X @ Wk, X @ Wv; dem_a += len(X)
    out_a = attn_cuoi(moi[i] @ Wq, K, V)

K_cache, V_cache = prompt @ Wk, prompt @ Wv; dem_b = len(prompt)
for i in range(50):
    K_cache = torch.cat([K_cache, (moi[i] @ Wk)[None]]); V_cache = torch.cat([V_cache, (moi[i] @ Wv)[None]]); dem_b += 1
    out_b = attn_cuoi(moi[i] @ Wq, K_cache, V_cache)
print(f"không cache: {dem_a} vector K | có cache: {dem_b} | giống nhau: {torch.allclose(out_a, out_b, atol=1e-5)}")

không cache: 6275 vector K | có cache: 150 | giống nhau: True


## Bài tập 6: Ước lượng tốc độ decode từ băng thông bộ nhớ {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
N = 8e9
BANG_THONG = 1e12
FLOPS = 200e12
BYTE = {"fp16": 2, "int8": 1, "int4": 0.5}
# TODO: tok_1 (dict định dạng → token/s cho 1 người); tok_batch16 (fp16, tổng token/s), gioi_han_tinh_toan

In [ ]:
tok_1 = {k: BANG_THONG / (N * b) for k, b in BYTE.items()}
def tong_tok(B, byte):
    t_buoc = max(N * byte / BANG_THONG, 2 * N * B / FLOPS)
    return B / t_buoc
tok_batch16 = tong_tok(16, 2)
gioi_han_tinh_toan = FLOPS / (2 * N)
print({k: round(v) for k, v in tok_1.items()})
print(f"batch 16 fp16: {tok_batch16:.0f} token/s tổng | trần tính toán: {gioi_han_tinh_toan:.0f} token/s")
print(f"batch bắt đầu chạm trần tính toán (fp16): khoảng {BYTE['fp16'] * FLOPS / (2 * BANG_THONG):.0f} người dùng")

{'fp16': 62, 'int8': 125, 'int4': 250}
batch 16 fp16: 1000 token/s tổng | trần tính toán: 12500 token/s
batch bắt đầu chạm trần tính toán (fp16): khoảng 200 người dùng


## Bài tập 7: Batch tĩnh và batch liên tục {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
do_dai = [20, 200, 35, 50, 180, 25, 40, 30]     # số token đầu ra của từng yêu cầu
B = 4
# TODO: buoc_tinh, buoc_lien_tuc, hieu_suat (tổng token / (số bước × B))

In [ ]:
buoc_tinh = sum(max(do_dai[i:i + B]) for i in range(0, len(do_dai), B))
hang, dang_chay, buoc_lien_tuc = list(do_dai), [], 0
while hang or dang_chay:
    while hang and len(dang_chay) < B:
        dang_chay.append(hang.pop(0))
    dang_chay = [r - 1 for r in dang_chay]
    dang_chay = [r for r in dang_chay if r > 0]
    buoc_lien_tuc += 1
for ten, s in [("tĩnh", buoc_tinh), ("liên tục", buoc_lien_tuc)]:
    print(f"{ten:9s}: {s} bước, dùng {sum(do_dai) / (s * B):.0%} chỗ trong batch")

tĩnh     : 380 bước, dùng 38% chỗ trong batch
liên tục : 200 bước, dùng 72% chỗ trong batch
